**Table of contents**<a id='toc0_'></a>    
- 1. [大模型的分布式推理概述](#toc1_)    
  - 1.1. [简介](#toc1_1_)    
  - 1.2. [案例场景：](#toc1_2_)    
- 2. [vLLM的分布式推理实现](#toc2_)    
- 3. [LMDeploy的分布式推理实现](#toc3_)    
- 4. [LMDeploy的量化机制](#toc4_)    
  - 4.1. [模型需要的显存计算](#toc4_1_)    
  - 4.2. [Key-Value(KV) Cache 量化](#toc4_2_)    
    - 4.2.1. [应用示例](#toc4_2_1_)    
    - 4.2.2. [离线推理](#toc4_2_2_)    
    - 4.2.3. [推理服务](#toc4_2_3_)    
    - 4.2.4. [精度评测](#toc4_2_4_)    
    - 4.2.5. [推理效率](#toc4_2_5_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[大模型的分布式推理概述](#toc0_)

## 1.1. <a id='toc1_1_'></a>[简介](#toc0_)

分布式推理是指将大模型的计算任务拆分到多个GPU设备上并行执行，以解决单卡显存不足、提升推理速度的技术。

其核心在于张量并行（Tensor Parallelism）和流水线并行（Pipeline Parallelism），其中张量并行将模型的权重矩阵按维度切分到不同GPU上，每个GPU负责部分计算，最终合并结果。

例如，在Llama-13B等大模型推理中，单卡显存可能不足，分布式推理可显著降低显存占用并提高吞吐量。


高迸发需要解决两个问题：显存不足、算力不足。

显存不足导致out of memory，算力不足导致推理速度变慢。

解决办法：更换性能更好的显卡或者增加显卡数量。

## 1.2. <a id='toc1_2_'></a>[案例场景：](#toc0_)

1.单卡显存不足：如QwQ-32B（320亿参数）需在双A6000显卡上部署。

2.高并发请求：在线服务需同时处理多用户请求，分布式推理通过连续批处理（Continuous Batching）提升效率。


# 2. <a id='toc2_'></a>[vLLM的分布式推理实现](#toc0_)

vLLM通过PagedAttention（分页注意力）和张量并行技术优化显存管理和计算效率，支持多GPU推理。

1. 核心机制
张量并行：通过tensor_parallel_size参数指定GPU数量，模型自动拆分到多卡。

PagedAttention：将注意力机制的键值（KV）缓存分块存储，减少显存碎片，提升利用率。

连续批处理：动态合并不同长度的请求，减少GPU空闲时间。


# 3. <a id='toc3_'></a>[LMDeploy的分布式推理实现](#toc0_)

LMDeploy在低性能显卡的推理上，性能是超过Vllm的，原因是LMDeploy在显存的把控和模型的实时量化上下的功夫是比较多的。

<img src="./Image/2025-05-10-10-15-15.png" style="margin-left: 0" width="60%">


LMDeploy是专为高效部署设计的框架，支持量化技术与分布式推理，尤其适合低显存环境。

1. 核心机制
张量并行：通过--tp参数指定GPU数量，支持多卡协同计算。

KV Cache量化：（Key-Value Cache 量化）支持INT8/INT4量化，降低显存占用。

动态显存管理：通过--cache-max-entry-count控制KV缓存比例。


# 4. <a id='toc4_'></a>[LMDeploy的量化机制](#toc0_)


## 4.1. <a id='toc4_1_'></a>[模型需要的显存计算](#toc0_)

所谓7B的模型B指的是10亿-billion，每个参数类型需要在模型的配置中查看，例如：bfloat16表示16位浮点数（bit）

![](Image/2025-05-10-10-55-14.png)

对于一个7B（70亿）参数的模型，每个参数使用16位浮点数（等于 2个 Byte）表示，则模型的权重大小约为：

70×10^9 parameters×2 Bytes/parameter=14GB

 70亿个参数×每个参数占用2个字节=14GB

这里的14GB是没有算其它程序占用、且没有人访问的情况下，所以我们需要大于14GB的显存。如果多人访问，假设模型输入的maxlength=8000，1bit=2byte，所以要X2，再乘以访问人数，8000*2*访问人数，然后Byte转换为GB

## 4.2. <a id='toc4_2_'></a>[Key-Value(KV) Cache 量化](#toc0_)

[Key-Value(KV) Cache 量化](https://lmdeploy.readthedocs.io/zh-cn/latest/quantization/kv_quant.html)

![](Image/2025-05-10-15-43-18.png)

### 4.2.1. <a id='toc4_2_1_'></a>[应用示例](#toc0_)

通过 LMDeploy 应用 kv 量化非常简单，只需要设定 quant_policy 参数。

LMDeploy 规定 qant_policy=4 表示 kv int4 量化，quant_policy=8 表示 kv int8 量化。

### 4.2.2. <a id='toc4_2_2_'></a>[离线推理](#toc0_)

from lmdeploy import pipeline, TurbomindEngineConfig  
engine_config = TurbomindEngineConfig(quant_policy=8)  
pipe = pipeline("internlm/internlm2_5-7b-chat", backend_config=engine_config)  
response = pipe(["Hi, pls intro yourself", "Shanghai is"])  
print(response)  

### 4.2.3. <a id='toc4_2_3_'></a>[推理服务](#toc0_)

lmdeploy serve api_server internlm/internlm2_5-7b-chat --quant-policy 8

此时模型还是16位的，但是推理的时候将16位量化到了8位，且精度损失很小。从显存占用上看不出来，因为显存占用还是框架决定的。但是推理速度会更快，也可以接受更多用户。

### 4.2.4. <a id='toc4_2_4_'></a>[精度评测](#toc0_)

我们把 lmdeploy 的 kv 量化应用在若干 LLM 模型上，并使用 opencompass 评测推理精度，结果如下表所示：

![](Image/2025-05-10-15-58-25.png) ![](Image/2025-05-10-15-59-11.png)

### 4.2.5. <a id='toc4_2_5_'></a>[推理效率](#toc0_)

![](Image/2025-05-10-16-03-15.png)